# NOVA 03 — MOD-05 Face Embedding (MobileFaceNet)
**Attach datasets:**
- `yakhyokhuja/vggface2-112x112` — VGGFace2 pre-aligned to 112x112 (exactly MobileFaceNet's input size; no alignment preprocessing needed)
- `jessicali9530/lfw-dataset` (evaluation only)

**Accelerator:** GPU. Training on a subset of identities ~6-10h.
Teacher (InsightFace ArcFace) downloads its weights on first run — internet must be ON in notebook settings.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q insightface onnxruntime-gpu onnx2tf onnx

In [ ]:
# Locate the identity-folder root inside the attached dataset
# (layout may nest one level — adjust after inspecting)
!find /kaggle/input/vggface2-112x112 -maxdepth 2 -type d | head -10
VGG_DATA = '/kaggle/input/vggface2-112x112/train'   # one folder per identity
!ls {VGG_DATA} | wc -l

In [ ]:
# MobileFaceNet + ArcFace loss + embedding-KD from InsightFace teacher.
# --max-identities 2000 keeps one run inside Kaggle's 12h session limit.
# Use --no-teacher to skip distillation if insightface weights fail to load.
!python scripts/train_face_embedding.py --data-dir {VGG_DATA} \
    --epochs 50 --batch-size 128 --max-identities 2000

In [ ]:
# Convert to TFLite INT8 (calibrate on a slice of training identities)
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/face_embedding_best.pth \
    --arch mobilefacenet --input-size 112 \
    --out /kaggle/working/exports/face_embedding_v1.tflite \
    --calib-dir {VGG_DATA} --benchmark

In [ ]:
# Publish to HuggingFace: unixio/nova-face-embedding
!python scripts/push_to_huggingface.py --module MOD-05-embed \
    --pytorch /kaggle/working/checkpoints/face_embedding_best.pth \
    --tflite /kaggle/working/exports/face_embedding_v1.tflite \
    --version 1.0.0